In [ ]:
!pip install torch_geometric
!pip install shodan
!pip install jsonlines

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data load

Load up hourly data from CAIDA stored in personal google drive

Don't want to wget everything because it's going to bother the authors!

In [ ]:
from pathlib import Path
import datetime
import pandas as pd

DATA_FOLDER = Path('/content/drive/MyDrive/DSCI599/data')

# glob all files ending with strategy
# hourly_files = list(DATA_FOLDER.glob("*.strategy"))
# only use one data for now, because it takes too long to use shodan to scan all
hourly_files = [DATA_FOLDER / "1464217200.strategy"]

hourly_columns = [
    "source_ip",
    "num_scanned_destination_ips",
    "num_unique_flows",
    "num_packets",
    "timestamp_first_activity",
    "timestamp_last_activity",
    "same_packet_size_flag",
    "avg_packet_size",
    "num_unique_Bs_scanned",
    "num_unique_Cs_scanned",
    "num_unique_Ds_scanned",
    "num_scanned_24_blocks",
    "num_non_conficker_destinations",
    "ip_address_range_diff",
    "destination_port",
    "protocol"
]

hourly_df = []
hourly_timestamps = []
for f in hourly_files:
  df = pd.read_csv(f, sep='|', header=None, names=hourly_columns)

  timestamp = int(f.stem)
  dt = str(datetime.datetime.fromtimestamp(timestamp))
  df.name = dt
  hourly_df.append(df)

  hourly_timestamps.append(dt)

In [ ]:
hourly_timestamps

['2016-05-25 23:00:00']

Store ip address as an int because it's hard to store without

> source: https://stackoverflow.com/questions/9590965/convert-an-ip-string-to-a-number-and-vice-versa

In [ ]:
import ipaddress

for i in range(len(hourly_df)):
  df = hourly_df[i]
  df['int_source_ip'] = df['source_ip'].map(lambda ip: int(ipaddress.ip_address(ip)))
  df = df.drop(columns = ['source_ip'])
  df = df.set_index('int_source_ip')
  hourly_df[i] = df

In [ ]:
hourly_df[0].head()

,num_scanned_destination_ips,num_unique_flows,num_packets,timestamp_first_activity,timestamp_last_activity,same_packet_size_flag,avg_packet_size,num_unique_Bs_scanned,num_unique_Cs_scanned,num_unique_Ds_scanned,num_scanned_24_blocks,num_non_conficker_destinations,ip_address_range_diff,destination_port,protocol
int_source_ip,,,,,,,,,,,,,,,
3065676419,161,161,490,1464218460,1464219240,1,60.0,118,112,124,161,39,16574121,23,6
3200595405,512,512,1045,1464220560,1464220620,1,29.0,2,2,256,2,128,13227007,53413,17
3153160574,255,255,255,1464217200,1464217440,1,60.0,1,1,255,1,128,254,23,6
2948693889,238,238,238,1464217200,1464220680,1,60.0,153,152,160,238,66,16205154,0,1
3383600711,255,255,255,1464218340,1464218580,1,60.0,1,1,255,1,128,254,23,6


# Parse Shodan JSON data

In [ ]:
import json
json_files = list(DATA_FOLDER.glob("*.json"))
json_files.remove(DATA_FOLDER / "shodan.json")
print(json_files)

[PosixPath('/content/drive/MyDrive/DSCI599/data/printers_10000.json')]


In [ ]:
from collections import defaultdict
iot_data = defaultdict(list)
for json_f in json_files:
  with open(json_f, 'r') as f:
    lines = f.readlines()

  parsed_lines = [json.loads(l) for l in lines]
  iot_data[json_f].extend(parsed_lines)
  print(parsed_lines[0].keys())

iot_ips = set()
for val in iot_data.values():
  for data in val:
    iot_ips.add(data.get('ip'))

dict_keys(['_shodan', 'asn', 'hash', 'os', 'timestamp', 'isp', 'transport', 'smb', 'hostnames', 'location', 'ip', 'domains', 'org', 'data', 'port', 'opts', 'ip_str'])


In [ ]:
len(iot_ips)

9645

In [ ]:
import numpy as np

uniq_ips = np.hstack([df.index.unique().values for df in hourly_df])
print(f"number of unique ip_addresses: {len(uniq_ips)}")

number of unique ip_addresses: 133829


# UNUSED

Originally, I used shodan as a label, but seems like that was wrong. So, I scrapped it.
## Shodan API

I mainly used online Shodan website to query but there's API usage as well if needed

In [ ]:
import shodan
from google.colab import userdata
SHODAN_API_KEY = userdata.get('SHODAN_API_KEY')

api = shodan.Shodan(SHODAN_API_KEY)

Shodan sample search
> documentation: https://shodan.readthedocs.io/en/latest/examples/basic-search.html

In [ ]:
import ipaddress
uniq_ips_str = [str(ipaddress.ip_address(int(x))) for x in uniq_ips]

Load up ips that have been checked

In [ ]:
with open(DATA_FOLDER / "checked.txt", 'r') as f:
  checked_history = f.read().strip().split("\n")

len(checked_history)

2277

In [ ]:
from tqdm import tqdm
dump_step = 100
data = []
checked = []
for ip in tqdm(uniq_ips_str):
  if ip in checked_history:
    continue

  checked.append(ip)

  try:
    res = api.host(ip)
    data.append(res)
  except Exception as e:
    continue

  if len(checked) % dump_step == 0:
    print(len(checked) % dump_step)
    # dump checked and data to not overflow RAM usage
    with open(DATA_FOLDER / "checked.txt", "a") as f:
      out_checked = "\n".join(checked) + "\n"
      f.write(out_checked)
    with jsonlines.open(DATA_FOLDER / "shodan.jsonl", mode="a") as writer:
      writer.write_all(data)

    data = []
    checked = []

  2%|▏         | 3240/133829 [29:50<54:44:54,  1.51s/it]

0


  4%|▍         | 5041/133829 [1:16:38<54:03:26,  1.51s/it]

0


  4%|▍         | 5341/133829 [1:23:57<46:34:37,  1.31s/it]

0


  5%|▍         | 6140/133829 [1:44:33<36:14:22,  1.02s/it]

0


UnicodeEncodeError: 'utf-8' codec can't encode character '\udfff' in position 1513: surrogates not allowed

In [ ]:
with open(DATA_FOLDER / "checked.txt", "a") as f:
  out_checked = "\n".join(checked) + "\n"
  f.write(out_checked)
with jsonlines.open(DATA_FOLDER / "shodan.jsonl", mode="a") as writer:
  writer.write_all(data)

In [ ]:
import jsonlines
json_data = [{"d": 1, "e": [2,3,4]}, {1:34565}]

with jsonlines.open(DATA_FOLDER / "test.jsonl", mode="a") as w:
  w.write_all(json_data)

## Checking for matching ips

In [ ]:
import numpy as np
matches = np.vectorize(lambda x: x in iot_ips)(uniq_ips)

In [ ]:
uniq_ips

array([3383606336, 2105501427, 3153160574, ..., 1479039867, 3077178347,
       1178514357])